In [10]:
#recovery part
# === RECOVERY===
from pathlib import Path
from collections import Counter
from tqdm import tqdm

ROOT      = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR  = ROOT / "data"
LBL_TRAIN = DATA_DIR / "labels" / "train"
LBL_VAL   = DATA_DIR / "labels" / "val"

# Step 1: Check current state
print("── Current state ──")
t = len(list(LBL_TRAIN.glob("*.txt")))
v = len(list(LBL_VAL.glob("*.txt")))
print(f"  train labels : {t}")
print(f"  val labels   : {v}")

ctr = Counter()
for ldir in [LBL_TRAIN, LBL_VAL]:
    for lf in ldir.glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                ctr[int(line.split()[0])] += 1
print(f"  indices found: {sorted(ctr.keys())}")
print(f"  max index    : {max(ctr.keys()) if ctr else 'N/A'}")

── Current state ──
  train labels : 10053
  val labels   : 3749
  indices found: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  max index    : 14


In [3]:
# Cell 1: Install Dependencies (with CUDA GPU support and extended timeout)

# 1. Install PyTorch with CUDA 12.1 (using a 1000-second timeout to prevent ReadTimeoutErrors)
!pip install --default-timeout=1000 torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. Install YOLO, Roboflow, and other required data processing libraries
!pip install ultralytics roboflow pycocotools opencv-python-headless matplotlib tqdm seaborn


Looking in indexes: https://download.pytorch.org/whl/cu121



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# project layout

In [13]:
# ======================================================================
# CELL 2 - Project Layout
# ======================================================================
from pathlib import Path

TRAINING_DIR = Path.cwd()
ROOT         = TRAINING_DIR.parent
DATA_DIR     = ROOT / "data"
ANNOT_DIR    = DATA_DIR / "annotations"

RAW_TRAIN = DATA_DIR / "training_data"
RAW_TEST  = DATA_DIR / "test_data"

IMG_TRAIN = DATA_DIR / "images" / "train"
IMG_VAL   = DATA_DIR / "images" / "val"
LBL_TRAIN = DATA_DIR / "labels" / "train"
LBL_VAL   = DATA_DIR / "labels" / "val"

OUTPUT_RUNS = ROOT / "runs"

# define function BEFORE calling it
def pick_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

BASE_WEIGHTS = pick_existing([
    TRAINING_DIR / "yolov8x.pt",
    ROOT / "yolov8x.pt",
    TRAINING_DIR / "yolov8n.pt",
    ROOT / "yolov8n.pt",
])

ANNOT_TRAIN_COCO = pick_existing([
    ANNOT_DIR / "training_coco.json",
    ANNOT_DIR / "train_coco.json",
    ANNOT_DIR / "coco_train.json",
])
ANNOT_TEST_COCO = pick_existing([
    ANNOT_DIR / "test_coco.json",
    ANNOT_DIR / "testing_coco.json",
    ANNOT_DIR / "coco_test.json",
])

print("ROOT             :", ROOT)
print("TRAINING_DIR     :", TRAINING_DIR)
print("RAW_TRAIN exists :", RAW_TRAIN.exists(), "->", len(list(RAW_TRAIN.iterdir())), "files")
print("RAW_TEST  exists :", RAW_TEST.exists(),  "->", len(list(RAW_TEST.iterdir())),  "files")
print("ANNOT_TRAIN_COCO :", ANNOT_TRAIN_COCO)
print("ANNOT_TEST_COCO  :", ANNOT_TEST_COCO)
print("BASE_WEIGHTS     :", BASE_WEIGHTS)

ROOT             : c:\Users\SHANTANU\flir_adas
TRAINING_DIR     : c:\Users\SHANTANU\flir_adas\training
RAW_TRAIN exists : True -> 10319 files
RAW_TEST  exists : True -> 3749 files
ANNOT_TRAIN_COCO : c:\Users\SHANTANU\flir_adas\data\annotations\training_coco.json
ANNOT_TEST_COCO  : c:\Users\SHANTANU\flir_adas\data\annotations\test_coco.json
BASE_WEIGHTS     : c:\Users\SHANTANU\flir_adas\yolov8x.pt


In [18]:
# === CELL 2b: LABEL RESET ===
from pathlib import Path
from tqdm import tqdm

ROOT      = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR  = ROOT / "data"
LBL_TRAIN = DATA_DIR / "labels" / "train"
LBL_VAL   = DATA_DIR / "labels" / "val"

# Delete all corrupted label files
for ldir in [LBL_TRAIN, LBL_VAL]:
    files = list(ldir.glob("*.txt"))
    for f in tqdm(files, desc=f"Deleting {ldir.name}"):
        f.unlink()
    print(f"Deleted {len(files)} files from {ldir.name}")

# Restore names.txt to full 80 classes so Cell 4 can remap correctly
# We just clear it — Cell 3 will rewrite it from COCO
(DATA_DIR / "names.txt").unlink(missing_ok=True)
print("Cleared names.txt")

t = len(list(LBL_TRAIN.glob("*.txt")))
v = len(list(LBL_VAL.glob("*.txt")))
print(f"\nLabels remaining — train:{t}  val:{v}  (must be 0)")
print("✓ Done — run Cell 3 next")

Deleting train: 100%|██████████| 10053/10053 [00:03<00:00, 3263.29it/s]


Deleted 10053 files from train


Deleting val: 100%|██████████| 3749/3749 [00:01<00:00, 2963.34it/s]

Deleted 3749 files from val
Cleared names.txt

Labels remaining — train:0  val:0  (must be 0)
✓ Done — run Cell 3 next


# coco -> yolo

In [19]:
# ======================================================================
# CELL 3 - COCO -> YOLO Label Conversion
# ======================================================================
import json
from pathlib import Path
from tqdm import tqdm

def find_disk_image(coco_fname, images_dir: Path):
    coco_fname = str(coco_fname)
    p_exact = images_dir / coco_fname
    if p_exact.exists():
        return p_exact
    stem = Path(coco_fname).stem
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]:
        p = images_dir / (stem + ext)
        if p.exists():
            return p
    for p in images_dir.iterdir():
        if p.is_file() and p.name.startswith(stem):
            return p
    for p in images_dir.iterdir():
        if p.is_file() and stem in p.name:
            return p
    return None


def coco_to_yolo_labels(coco_json_path, images_dir, out_labels_dir):
    out_labels_dir.mkdir(parents=True, exist_ok=True)
    with open(coco_json_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    images      = {im["id"]: im for im in coco.get("images", [])}
    annotations = coco.get("annotations", [])
    categories  = {cat["id"]: cat["name"] for cat in coco.get("categories", [])}
    cat_ids     = sorted(categories.keys())
    cat_to_idx  = {cid: i for i, cid in enumerate(cat_ids)}

    written = set()
    missing = set()
    skipped = 0

    for ann in tqdm(annotations, desc=f"Converting {coco_json_path.name}"):
        img_id = ann.get("image_id")
        img    = images.get(img_id)
        if img is None:
            skipped += 1
            continue

        file_name    = img.get("file_name")
        img_w, img_h = img.get("width"), img.get("height")
        disk_path    = find_disk_image(file_name, images_dir)

        if disk_path is None:
            missing.add(file_name)
            continue

        bbox = ann.get("bbox", [])
        if len(bbox) != 4:
            skipped += 1
            continue
        cid = ann.get("category_id")
        if cid not in cat_to_idx:
            skipped += 1
            continue

        x, y, w, h = bbox
        x_c   = (x + w / 2.0) / img_w
        y_c   = (y + h / 2.0) / img_h
        w_rel = w / img_w
        h_rel = h / img_h
        cls   = cat_to_idx[cid]

        out_path = out_labels_dir / disk_path.with_suffix(".txt").name
        with open(out_path, "a", encoding="utf-8") as f:
            f.write(f"{cls} {x_c:.6f} {y_c:.6f} {w_rel:.6f} {h_rel:.6f}\n")
        written.add(out_path.name)

    print(f"  labels written        : {len(written)}")
    print(f"  missing images        : {len(missing)}")
    print(f"  skipped annotations   : {skipped}")
    print(f"  total annotations     : {len(annotations)}")
    return written


# Only convert if labels don't already exist
if not LBL_TRAIN.exists() or len(list(LBL_TRAIN.glob("*.txt"))) == 0:
    if ANNOT_TRAIN_COCO:
        print("Converting TRAIN...")
        coco_to_yolo_labels(ANNOT_TRAIN_COCO, RAW_TRAIN, LBL_TRAIN)
    else:
        print("No TRAIN COCO found.")
else:
    print(f"TRAIN labels already exist: {len(list(LBL_TRAIN.glob('*.txt')))} files — skipping conversion")

if not LBL_VAL.exists() or len(list(LBL_VAL.glob("*.txt"))) == 0:
    if ANNOT_TEST_COCO:
        print("Converting VAL...")
        coco_to_yolo_labels(ANNOT_TEST_COCO, RAW_TEST, LBL_VAL)
    else:
        print("No TEST COCO found.")
else:
    print(f"VAL labels already exist: {len(list(LBL_VAL.glob('*.txt')))} files — skipping conversion")

# Save names.txt
src_coco   = ANNOT_TRAIN_COCO or ANNOT_TEST_COCO
names_file = DATA_DIR / "names.txt"
if src_coco and src_coco.exists():
    with open(src_coco, "r", encoding="utf-8") as f:
        cats = json.load(f).get("categories", [])
    names = [c["name"] for c in sorted(cats, key=lambda x: x["id"])]
    names_file.write_text("\n".join(names), encoding="utf-8")
    print(f"Saved {len(names)} class names -> {names_file}")
else:
    raise FileNotFoundError("No COCO file found to extract class names.")


Converting TRAIN...


Converting training_coco.json: 100%|██████████| 169174/169174 [02:04<00:00, 1355.85it/s]


  labels written        : 10053
  missing images        : 0
  skipped annotations   : 0
  total annotations     : 169174
Converting VAL...


Converting test_coco.json: 100%|██████████| 84786/84786 [01:58<00:00, 715.50it/s] 


  labels written        : 3749
  missing images        : 0
  skipped annotations   : 0
  total annotations     : 84786
Saved 80 class names -> c:\Users\SHANTANU\flir_adas\data\names.txt


In [20]:
# ======================================================================
# CELL 4 - Remap Class Indices (remove unused classes)
# ======================================================================
from pathlib import Path
from collections import Counter
from tqdm import tqdm

names_all = (DATA_DIR / "names.txt").read_text().strip().splitlines()

# Scan all label files to find used class indices
counter = Counter()
for ldir in [LBL_TRAIN, LBL_VAL]:
    for lf in ldir.glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1

used_indices = sorted(counter.keys())
new_names    = [names_all[i] for i in used_indices]
old_to_new   = {old: new for new, old in enumerate(used_indices)}

print(f"Classes used: {len(used_indices)} of {len(names_all)}")
print(f"\n{'old':>5}  {'new':>5}  {'name':<22}  {'count':>8}")
print("-" * 48)
for old, new in old_to_new.items():
    print(f"{old:>5}  {new:>5}  {names_all[old]:<22}  {counter[old]:>8}")

# Rewrite label files with remapped indices
def remap_labels(labels_dir, old_to_new, split):
    files = list(labels_dir.glob("*.txt"))
    for lf in tqdm(files, desc=f"Remapping {split}"):
        lines     = lf.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.split()
            if not parts:
                continue
            old_cls = int(parts[0])
            if old_cls not in old_to_new:
                continue
            parts[0] = str(old_to_new[old_cls])
            new_lines.append(" ".join(parts))
        lf.write_text("\n".join(new_lines) + "\n", encoding="utf-8")

remap_labels(LBL_TRAIN, old_to_new, "train")
remap_labels(LBL_VAL,   old_to_new, "val")

# Update names.txt
names = new_names
(DATA_DIR / "names.txt").write_text("\n".join(names), encoding="utf-8")
print(f"\nFinal classes ({len(names)}): {names}")


Classes used: 15 of 80

  old    new  name                       count
------------------------------------------------
    0      0  person                     46285
    1      1  bike                        8006
    2      2  car                       102174
    3      3  motor                       5410
    5      4  bus                         1879
    6      5  train                          9
    7      6  truck                       2750
    9      7  light                      36458
   10      8  hydrant                     1392
   11      9  sign                       46744
   16     10  dog                           17
   36     11  skateboard                   412
   72     12  stroller                      38
   74     13  scooter                       41
   78     14  other vehicle               2345


Remapping val: 100%|██████████| 3749/3749 [00:02<00:00, 1651.18it/s]


Final classes (15): ['person', 'bike', 'car', 'motor', 'bus', 'train', 'truck', 'light', 'hydrant', 'sign', 'dog', 'skateboard', 'stroller', 'scooter', 'other vehicle']


In [21]:
# ======================================================================
# CELL 5 - Fix Image Folders (remove junctions, create real dirs)
# ======================================================================
import os, ctypes, shutil
from pathlib import Path
from tqdm import tqdm

def is_junction(p: Path) -> bool:
    try:
        attrs = ctypes.windll.kernel32.GetFileAttributesW(str(p))
        return bool(attrs & 0x400)
    except Exception:
        return False

def remove_junction_or_dir(p: Path):
    if not p.exists() and not p.is_symlink():
        print(f"  Does not exist (ok): {p.name}")
        return
    if is_junction(p):
        os.rmdir(p)
        print(f"  Removed junction   : {p.name}")
    else:
        shutil.rmtree(p)
        print(f"  Removed real dir   : {p.name}")

def hardlink_images(src: Path, dst: Path, label: str):
    exts  = {".jpg", ".jpeg", ".png"}
    imgs  = [p for p in src.iterdir() if p.suffix.lower() in exts]
    count = 0
    for img in tqdm(imgs, desc=f"  Linking {label}"):
        dst_file = dst / img.name
        if not dst_file.exists():
            try:
                os.link(img, dst_file)
            except OSError:
                shutil.copy2(img, dst_file)
            count += 1
    print(f"  {label}: {count} images linked -> {dst}")

# Remove old junctions
print("── Step 1: Remove junctions ──")
remove_junction_or_dir(IMG_TRAIN)
remove_junction_or_dir(IMG_VAL)

# Create real directories
print("\n── Step 2: Create real dirs ──")
IMG_TRAIN.mkdir(parents=True, exist_ok=True)
IMG_VAL.mkdir(parents=True, exist_ok=True)
print(f"  Created: {IMG_TRAIN}")
print(f"  Created: {IMG_VAL}")

# Hardlink images
print("\n── Step 3: Hardlink images ──")
hardlink_images(RAW_TRAIN, IMG_TRAIN, "TRAIN")
hardlink_images(RAW_TEST,  IMG_VAL,   "VAL")

# Verify
print("\n── Step 4: Verify ──")
for p, label in [(IMG_TRAIN, "TRAIN"), (IMG_VAL, "VAL")]:
    junc     = is_junction(p)
    resolved = str(p.resolve()).replace("\\", "/")
    count    = len(list(p.iterdir()))
    print(f"  {label}")
    print(f"    is_junction : {junc}      <- must be False")
    print(f"    resolves to : {resolved}  <- must contain 'images'")
    print(f"    file count  : {count}")
    assert not junc,                  f"Junction still active: {p}"
    assert "images" in resolved,      f"Wrong resolved path: {resolved}"
    assert count > 0,                 f"No images in: {p}"

print("\n✓ Image folders are real directories")


── Step 1: Remove junctions ──
  Removed real dir   : train
  Removed real dir   : val

── Step 2: Create real dirs ──
  Created: c:\Users\SHANTANU\flir_adas\data\images\train
  Created: c:\Users\SHANTANU\flir_adas\data\images\val

── Step 3: Hardlink images ──


  Linking TRAIN: 100%|██████████| 10319/10319 [00:16<00:00, 613.34it/s]


  TRAIN: 10319 images linked -> c:\Users\SHANTANU\flir_adas\data\images\train


  Linking VAL: 100%|██████████| 3749/3749 [00:05<00:00, 682.72it/s]


  VAL: 3749 images linked -> c:\Users\SHANTANU\flir_adas\data\images\val

── Step 4: Verify ──
  TRAIN
    is_junction : False      <- must be False
    resolves to : C:/Users/SHANTANU/flir_adas/data/images/train  <- must contain 'images'
    file count  : 10319
  VAL
    is_junction : False      <- must be False
    resolves to : C:/Users/SHANTANU/flir_adas/data/images/val  <- must contain 'images'
    file count  : 3749

✓ Image folders are real directories


In [22]:
# ======================================================================
# CELL 6 - Verify Everything Before Training
# ======================================================================
from pathlib import Path
from collections import Counter

# 1. Stem matching
def check_stems(imgs_dir, lbls_dir, split):
    imgs   = {p.stem for p in imgs_dir.glob("*")
              if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
    labels = {p.stem for p in lbls_dir.glob("*.txt")}
    matched = imgs & labels
    print(f"\n── {split} ──────────────────────────────")
    print(f"  images  : {len(imgs)}")
    print(f"  labels  : {len(labels)}")
    print(f"  matched : {len(matched)}  <- must be > 0")
    print(f"  backgrounds (no label): {len(imgs - labels)}")
    assert len(matched) > 0, f"0 matched pairs in {split}!"
    return len(matched)

check_stems(IMG_TRAIN, LBL_TRAIN, "TRAIN")
check_stems(IMG_VAL,   LBL_VAL,   "VAL")

# 2. YOLO path swap check
names  = (DATA_DIR / "names.txt").read_text().strip().splitlines()
train_resolved = str(IMG_TRAIN.resolve()).replace("\\", "/")
lbl_via_swap   = train_resolved.replace("images", "labels")
lbl_actual     = str(LBL_TRAIN.resolve()).replace("\\", "/")

print(f"\n── YOLO label discovery check ──────────")
print(f"  YOLO swaps path to : {lbl_via_swap}")
print(f"  Labels actually at : {lbl_actual}")
print(f"  Match              : {lbl_via_swap == lbl_actual}  <- must be True")
assert lbl_via_swap == lbl_actual, (
    f"YOLO will not find labels!\n"
    f"  Expected: {lbl_via_swap}\n"
    f"  Actual  : {lbl_actual}"
)

# 3. Class index check
ctr = Counter()
for ldir in [LBL_TRAIN, LBL_VAL]:
    for lf in ldir.glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                ctr[int(line.split()[0])] += 1
max_idx = max(ctr.keys())
print(f"\n── Class check ─────────────────────────")
print(f"  nc        : {len(names)}")
print(f"  max index : {max_idx}  <- must be < nc")
assert max_idx < len(names), f"Class index {max_idx} out of range for nc={len(names)}"

print("\n✓ ALL CHECKS PASSED — safe to write data.yaml and train")



── TRAIN ──────────────────────────────
  images  : 10319
  labels  : 10053
  matched : 10053  <- must be > 0
  backgrounds (no label): 266

── VAL ──────────────────────────────
  images  : 3749
  labels  : 3749
  matched : 3749  <- must be > 0
  backgrounds (no label): 0

── YOLO label discovery check ──────────
  YOLO swaps path to : C:/Users/SHANTANU/flir_adas/data/labels/train
  Labels actually at : C:/Users/SHANTANU/flir_adas/data/labels/train
  Match              : True  <- must be True

── Class check ─────────────────────────
  nc        : 15
  max index : 14  <- must be < nc

✓ ALL CHECKS PASSED — safe to write data.yaml and train


In [23]:
# ======================================================================
# CELL 7 - Write data.yaml
# ======================================================================
from pathlib import Path

names     = (DATA_DIR / "names.txt").read_text().strip().splitlines()
TRAIN_ABS = str(IMG_TRAIN.resolve()).replace("\\", "/")
VAL_ABS   = str(IMG_VAL.resolve()).replace("\\", "/")

# Delete all stale caches
for c in ROOT.rglob("*.cache"):
    c.unlink()
    print(f"Deleted cache: {c.name}")

# Write yaml as plain text — no library, no formatting surprises
name_lines = "\n".join(f"  - {n}" for n in names)
yaml_text  = (
    f"train: {TRAIN_ABS}\n"
    f"val: {VAL_ABS}\n"
    f"nc: {len(names)}\n"
    f"names:\n{name_lines}\n"
)

yaml_path = TRAINING_DIR / "data.yaml"
yaml_path.write_text(yaml_text, encoding="utf-8")

print("\n✓ data.yaml written:")
print(yaml_path.read_text())

# Confirm
assert "images/train" in yaml_path.read_text(), "images/train not in yaml!"
assert "images/val"   in yaml_path.read_text(), "images/val not in yaml!"
print("✓ data.yaml verified")


✓ data.yaml written:
train: C:/Users/SHANTANU/flir_adas/data/images/train
val: C:/Users/SHANTANU/flir_adas/data/images/val
nc: 15
names:
  - person
  - bike
  - car
  - motor
  - bus
  - train
  - truck
  - light
  - hydrant
  - sign
  - dog
  - skateboard
  - stroller
  - scooter
  - other vehicle

✓ data.yaml verified


In [24]:
# ======================================================================
# CELL 8 - Train
# ======================================================================
from ultralytics import YOLO
from pathlib import Path
import torch

# Final cache sweep
for c in ROOT.rglob("*.cache"):
    c.unlink()

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {round(torch.cuda.get_device_properties(0).total_memory/1e9,1)} GB")

# Confirm yaml one last time
yaml_content = (TRAINING_DIR / "data.yaml").read_text()
print(f"\ndata.yaml preview:\n{yaml_content[:300]}")
assert "images/train" in yaml_content, "WRONG yaml — re-run Cell 7"

model = YOLO(str(BASE_WEIGHTS))
model.train(
    data      = str(TRAINING_DIR / "data.yaml"),
    epochs    = 50,
    imgsz     = 640,
    batch     = 2,
    device    = 0,
    project   = str(OUTPUT_RUNS),
    name      = "yolov8x_flir",
    exist_ok  = True,
    workers   = 2,
    amp       = True,
    cache     = False,
    close_mosaic = 10,
)

best = OUTPUT_RUNS / "yolov8x_flir" / "weights" / "best.pt"
print(f"\n✓ Training complete. Best weights: {best}")
assert best.exists(), "best.pt not found — training may have failed"


GPU  : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM : 6.4 GB

data.yaml preview:
train: C:/Users/SHANTANU/flir_adas/data/images/train
val: C:/Users/SHANTANU/flir_adas/data/images/val
nc: 15
names:
  - person
  - bike
  - car
  - motor
  - bus
  - train
  - truck
  - light
  - hydrant
  - sign
  - dog
  - skateboard
  - stroller
  - scooter
  - other vehicle



c:\Users\SHANTANU\flir_adas\flir\Lib\site-packages\ultralytics\nn\tasks.py:732: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(file, map_location="cpu")


New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.27  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: task=detect, mode=train, model=c:\Users\SHANTANU\flir_adas\yolov8x.pt, data=c:\Users\SHANTANU\flir_adas\training\data.yaml, epochs=50, time=None, patience=100, batch=2, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=c:\Users\SHANTANU\flir_adas\runs, name=yolov8x_flir, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=Fa

c:\Users\SHANTANU\flir_adas\flir\Lib\site-packages\ultralytics\nn\tasks.py:732: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(file, map_location="cpu")
c:\

AMP: checks passed 


...

                   all       3749      84785      0.492       0.28      0.313      0.178

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      14/50      3.15G      1.078     0.7099     0.9931          7        640: 100%|██████████| 5160/5160 [1:36:12<00:00,  1.12s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 938/938 [10:05<00:00,  1.55it/s]


                   all       3749      84785      0.508      0.292      0.323      0.182

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      15/50      3.11G      1.073     0.7069     0.9846         44        640:  35%|███▍      | 1791/5160 [35:00<1:05:52,  1.17s/it]


KeyboardInterrupt: 

In [4]:
# ======================================================================
# CELL 8b - Resume Training with batch=6
# ======================================================================
from ultralytics import YOLO
from pathlib import Path

ROOT         = Path(r"c:\Users\SHANTANU\flir_adas")
TRAINING_DIR = ROOT / "training"
OUTPUT_RUNS  = ROOT / "runs"

model = YOLO(str(OUTPUT_RUNS / "yolov8x_flir" / "weights" / "last.pt"))
model.train(
    data         = str(TRAINING_DIR / "data.yaml"),
    epochs       = 50,
    imgsz        = 640,
    batch        = 3,
    device       = 0,
    project      = str(OUTPUT_RUNS),
    name         = "yolov8x_flir",
    exist_ok     = True,
    workers      = 0,
    amp          = True,
    cache        = False,
    resume       = True,
    close_mosaic = 10,
)

best = OUTPUT_RUNS / "yolov8x_flir" / "weights" / "best.pt"
print(f"\n✓ Done. Best weights: {best}")

Ultralytics YOLOv8.2.27  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: task=detect, mode=train, model=c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\weights\last.pt, data=c:\Users\SHANTANU\flir_adas\training\data.yaml, epochs=50, time=None, patience=100, batch=3, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=c:\Users\SHANTANU\flir_adas\runs, name=yolov8x_flir, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\weights\last.pt, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, a

train: Scanning C:\Users\SHANTANU\flir_adas\data\labels\train.cache... 10053 images, 266 backgrounds, 0 corrupt: 100%|██████████| 10319/10319 [00:00<?, ?it/s]
val: Scanning C:\Users\SHANTANU\flir_adas\data\labels\val.cache... 3749 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3749/3749 [00:00<?, ?it/s]

val: WARNING  C:\Users\SHANTANU\flir_adas\data\images\val\video-RMxN6a4CcCeLGu4tA-frame-000207-y5eHLa6XNJkpcXEAS.jpg: 1 duplicate labels removed


Plotting labels to c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000526, momentum=0.9) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0004921875), 103 bias(decay=0.0)
Resuming training c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\weights\last.pt from epoch 15 to 50 total epochs
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      15/50       5.3G      1.071     0.7026     0.9918         60        640: 100%|██████████| 3440/3440 [1:41:57<00:00,  1.78s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 625/625 [12:06<00:00,  1.16s/it]


                   all       3749      84785      0.497      0.277      0.319      0.177

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      16/50      5.34G      1.061     0.6854     0.9861         26        640: 100%|██████████| 3440/3440 [1:34:22<00:00,  1.65s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 625/625 [12:32<00:00,  1.20s/it]


                   all       3749      84785      0.505      0.273       0.31      0.174


ValueError: I/O operation on closed file.

In [5]:
import shutil
from pathlib import Path

ROOT = Path(r"c:\Users\SHANTANU\flir_adas")

# Check disk space
total, used, free = shutil.disk_usage(ROOT)
print(f"Total : {total/1e9:.1f} GB")
print(f"Used  : {used/1e9:.1f} GB")
print(f"Free  : {free/1e9:.1f} GB")

# Check weights files
weights_dir = ROOT / "runs" / "yolov8x_flir" / "weights"
print(f"\nWeights files:")
for w in weights_dir.glob("*.pt"):
    size = w.stat().st_size / 1e6
    print(f"  {w.name}  {size:.1f} MB")

Total : 196.5 GB
Used  : 190.2 GB
Free  : 6.3 GB

Weights files:
  best.pt  409.7 MB
  last.pt  409.7 MB


In [6]:
import shutil
from pathlib import Path

ROOT    = Path(r"c:\Users\SHANTANU\flir_adas")
NEW_DIR = Path(r"D:\flir_adas_runs")  # change to your D: drive path

# Check D: drive space first
total, used, free = shutil.disk_usage("D:\\")
print(f"D: drive free: {free/1e9:.1f} GB")

# Move runs folder to D:
src = ROOT / "runs"
dst = NEW_DIR / "runs"

print(f"\nMoving {src} -> {dst}")
shutil.copytree(src, dst)
print("Copy complete")

# Verify
print(f"\nWeights in new location:")
for w in (dst / "yolov8x_flir" / "weights").glob("*.pt"):
    print(f"  {w.name}  {w.stat().st_size/1e6:.1f} MB")

D: drive free: 253.1 GB

Moving c:\Users\SHANTANU\flir_adas\runs -> D:\flir_adas_runs\runs
Copy complete

Weights in new location:
  best.pt  409.7 MB
  last.pt  409.7 MB


In [4]:
from ultralytics import YOLO
from pathlib import Path

TRAINING_DIR = Path(r"c:\Users\SHANTANU\flir_adas\training")
NEW_RUNS     = Path(r"D:\flir_adas_runs\runs")

model = YOLO(str(NEW_RUNS / "yolov8x_flir" / "weights" / "last.pt"))

model.train(
    data         = str(TRAINING_DIR / "data.yaml"),
    epochs       = 50,
    imgsz        = 640,
    batch        = 3,
    device       = 0,
    project      = str(NEW_RUNS),
    name         = "yolov8x_flir",
    exist_ok     = True,
    workers      = 0,
    amp          = True,
    cache        = False,
    resume       = True,
    close_mosaic = 10,
)

best = NEW_RUNS / "yolov8x_flir" / "weights" / "best.pt"
print(f"Done. Best weights: {best}")

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.27  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: task=detect, mode=train, model=D:\flir_adas_runs\runs\yolov8x_flir\weights\last.pt, data=c:\Users\SHANTANU\flir_adas\training\data.yaml, epochs=50, time=None, patience=100, batch=3, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=c:\Users\SHANTANU\flir_adas\runs, name=yolov8x_flir, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=D:\flir_adas_runs\runs\yolov8x_flir\weights\last.pt, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, s

train: Scanning C:\Users\SHANTANU\flir_adas\data\labels\train.cache... 10053 images, 266 backgrounds, 0 corrupt: 100%|██████████| 10319/10319 [00:00<?, ?it/s]
val: Scanning C:\Users\SHANTANU\flir_adas\data\labels\val.cache... 3749 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3749/3749 [00:00<?, ?it/s]

val: WARNING  C:\Users\SHANTANU\flir_adas\data\images\val\video-RMxN6a4CcCeLGu4tA-frame-000207-y5eHLa6XNJkpcXEAS.jpg: 1 duplicate labels removed


Plotting labels to c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000526, momentum=0.9) with parameter groups 97 weight(decay=0.0), 104 weight(decay=0.0004921875), 103 bias(decay=0.0)
Resuming training D:\flir_adas_runs\runs\yolov8x_flir\weights\last.pt from epoch 46 to 50 total epochs
Closing dataloader mosaic
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      46/50      5.17G     0.8989     0.5047     0.9127         23        640: 100%|██████████| 3440/3440 [1:37:36<00:00,  1.70s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 625/625 [12:36<00:00,  1.21s/it]


...

                   all       3749      84785      0.518      0.315      0.348      0.203

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      50/50      5.23G     0.8786     0.4831     0.9058         38        640: 100%|██████████| 3440/3440 [1:38:54<00:00,  1.73s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 625/625 [11:02<00:00,  1.06s/it]


                   all       3749      84785      0.512      0.314      0.346      0.202

5 epochs completed in 9.557 hours.
Optimizer stripped from c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\weights\last.pt, 136.7MB
Optimizer stripped from c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\weights\best.pt, 136.7MB

Validating c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\weights\best.pt...
Ultralytics YOLOv8.2.27  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 268 layers, 68138013 parameters, 0 gradients, 257.5 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 625/625 [11:21<00:00,  1.09s/it]


                   all       3749      84785      0.515      0.348      0.367      0.205
                person       2233      11278      0.651      0.596      0.634      0.362
                  bike        263        446      0.227      0.428      0.232      0.113
                   car       3614      30892      0.649      0.667      0.701      0.486
                 motor        989       3573      0.597      0.573      0.582      0.289
                 truck        791       1499      0.358       0.22      0.152     0.0448
                 light       1441      17818      0.637      0.244      0.292      0.125
               hydrant        309        402       0.83      0.231      0.426      0.239
                  sign       2140      17213      0.558      0.419      0.432      0.261
                   dog         17         17          0          0          0          0
         other vehicle        505       1647      0.647      0.103      0.221      0.135
Speed: 0.6ms preproce

In [5]:
import shutil, torch
from pathlib import Path

C_WEIGHTS = Path(r"c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\weights")
D_WEIGHTS = Path(r"D:\flir_adas_runs\runs\yolov8x_flir\weights")

# Copy final best.pt to D:
shutil.copy2(C_WEIGHTS / "best.pt", D_WEIGHTS / "best.pt")
shutil.copy2(C_WEIGHTS / "last.pt", D_WEIGHTS / "last.pt")

# Verify
ckpt = torch.load(str(D_WEIGHTS / "best.pt"), map_location="cpu")
print(f"best.pt epoch : {ckpt.get('epoch')}")
print(f"Copied to D: successfully")

best.pt epoch : -1
Copied to D: successfully


In [6]:
from ultralytics import YOLO
from pathlib import Path

ROOT         = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR     = ROOT / "data"
TRAINING_DIR = ROOT / "training"
NEW_RUNS     = Path(r"D:\flir_adas_runs\runs")

# Delete old caches
for c in DATA_DIR.rglob("*.cache"):
    c.unlink()
    print(f"Deleted cache: {c.name}")

model = YOLO("yolov8s.pt")  # auto-downloads if not present
model.train(
    data         = str(TRAINING_DIR / "data.yaml"),
    epochs       = 100,
    imgsz        = 640,
    batch        = 8,
    device       = 0,
    project      = str(NEW_RUNS),
    name         = "yolov8s_flir",
    exist_ok     = True,
    workers      = 2,
    amp          = True,
    cache        = False,
    close_mosaic = 10,
    cls          = 1.0,
    box          = 9.0,
)

best = NEW_RUNS / "yolov8s_flir" / "weights" / "best.pt"
print(f"Done. Best weights: {best}")

Deleted cache: train.cache
Deleted cache: val.cache


100%|██████████| 21.5M/21.5M [00:24<00:00, 928kB/s] 


New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.27  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: task=detect, mode=train, model=yolov8s.pt, data=c:\Users\SHANTANU\flir_adas\training\data.yaml, epochs=100, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=2, project=D:\flir_adas_runs\runs, name=yolov8s_flir, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=Fals

train: Scanning C:\Users\SHANTANU\flir_adas\data\labels\train... 10053 images, 266 backgrounds, 0 corrupt: 100%|██████████| 10319/10319 [00:32<00:00, 316.33it/s]


train: New cache created: C:\Users\SHANTANU\flir_adas\data\labels\train.cache


...

                   all       3749      84785      0.499      0.225      0.274      0.151

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     39/100      4.18G      1.341      1.371     0.9462        143        640: 100%|██████████| 1290/1290 [19:20<00:00,  1.11it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 235/235 [03:51<00:00,  1.01it/s]


                   all       3749      84785      0.494       0.22      0.271      0.151

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     40/100      4.32G      1.336      1.362     0.9486        181        640: 100%|██████████| 1290/1290 [19:27<00:00,  1.10it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):  14%|█▎        | 32/235 [00:30<03:16,  1.03it/s]


KeyboardInterrupt: 

In [ ]:
from pathlib import Path

LBL = Path(r"c:\Users\SHANTANU\flir_adas\data\labels\train")
IMG = Path(r"c:\Users\SHANTANU\flir_adas\data\images\train")

aug_labels = list(LBL.glob("aug_*.txt"))
aug_images = list(IMG.glob("aug_*.jpg"))

print(f"Aug label files : {len(aug_labels)}")
print(f"Aug image files : {len(aug_images)}")

In [5]:
import os, shutil
from pathlib import Path
from tqdm import tqdm

ROOT      = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR  = ROOT / "data"

RAW_TRAIN = DATA_DIR / "training_data"
RAW_TEST  = DATA_DIR / "test_data"
IMG_TRAIN = DATA_DIR / "images" / "train"
IMG_VAL   = DATA_DIR / "images" / "val"

# Recreate directories
IMG_TRAIN.mkdir(parents=True, exist_ok=True)
IMG_VAL.mkdir(parents=True, exist_ok=True)

# Hardlink images back
def hardlink_images(src, dst, label):
    imgs  = [p for p in src.iterdir() if p.suffix.lower() in {".jpg",".jpeg",".png"}]
    count = 0
    for img in tqdm(imgs, desc=f"Linking {label}"):
        dst_file = dst / img.name
        if not dst_file.exists():
            try:
                os.link(img, dst_file)
            except OSError:
                shutil.copy2(img, dst_file)
            count += 1
    print(f"  {label}: {count} files -> {dst}")

print("Recreating image folders...")
hardlink_images(RAW_TRAIN, IMG_TRAIN, "TRAIN")
hardlink_images(RAW_TEST,  IMG_VAL,   "VAL")
print("✓ Done — now resume training")

Recreating image folders...


Linking TRAIN: 100%|██████████| 10319/10319 [00:19<00:00, 519.00it/s]


  TRAIN: 10319 files -> c:\Users\SHANTANU\flir_adas\data\images\train


Linking VAL: 100%|██████████| 3749/3749 [00:06<00:00, 556.87it/s]

  VAL: 3749 files -> c:\Users\SHANTANU\flir_adas\data\images\val
✓ Done — now resume training


In [3]:
import torch, shutil
from pathlib import Path

C_WEIGHTS = Path(r"c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\weights")
D_WEIGHTS = Path(r"D:\flir_adas_runs\runs\yolov8x_flir\weights")

# Check which last.pt is newer
c_epoch = torch.load(str(C_WEIGHTS / "last.pt"), map_location="cpu").get("epoch")
d_epoch = torch.load(str(D_WEIGHTS / "last.pt"), map_location="cpu").get("epoch")
print(f"C: last.pt epoch: {c_epoch}")
print(f"D: last.pt epoch: {d_epoch}")

# Copy the newer one to D:
if c_epoch > d_epoch:
    shutil.copy2(C_WEIGHTS / "last.pt", D_WEIGHTS / "last.pt")
    shutil.copy2(C_WEIGHTS / "best.pt", D_WEIGHTS / "best.pt")
    print(f"Copied C: epoch {c_epoch} -> D:")
else:
    print(f"D: already has latest epoch {d_epoch}")

C: last.pt epoch: 44
D: last.pt epoch: 28
Copied C: epoch 44 -> D:


In [7]:
from pathlib import Path
import pandas as pd

ROOT        = Path(r"c:\Users\SHANTANU\flir_adas")
results_csv = ROOT / "runs" / "yolov8x_flir" / "results.csv"

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
print(f"Total epochs logged: {len(df)}")
print(df[["epoch", "metrics/mAP50(B)"]].to_string())

Total epochs logged: 51
    epoch  metrics/mAP50(B)
0       1           0.22768
1       2           0.24744
2       3           0.23243
3       4           0.26835
4       5           0.28164
5       6           0.27576
6       7           0.29393
7       8           0.31055
8       9           0.31119
9      10           0.29739
10     11           0.31740
11     12           0.29888
12     13           0.31290
13     14           0.32301
14     15           0.31854
15     16           0.30955
16     16           0.32163
17     17           0.31369
18     18           0.32451
19     19           0.32680
20     20           0.33894
21     21           0.36046
22     22           0.33628
23     23           0.35066
24     24           0.33745
25     25           0.33632
26     26           0.34625
27     27           0.34963
28     28           0.35308
29     29           0.34642
30     30           0.36157
31     31           0.34836
32     32           0.33175
33     33           0.34

In [4]:
from pathlib import Path

C_RUNS = Path(r"c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir")
D_RUNS = Path(r"D:\flir_adas_runs\runs\yolov8x_flir")

print("── C: drive ──")
for f in C_RUNS.rglob("*"):
    if f.is_file():
        print(f"  {f.relative_to(C_RUNS)}  {f.stat().st_size/1e6:.1f} MB")

print("\n── D: drive ──")
for f in D_RUNS.rglob("*"):
    if f.is_file():
        print(f"  {f.relative_to(D_RUNS)}  {f.stat().st_size/1e6:.1f} MB")

── C: drive ──
  args.yaml  0.0 MB
  labels.jpg  0.1 MB
  labels_correlogram.jpg  0.2 MB
  results.csv  0.0 MB
  train_batch0.jpg  0.1 MB
  train_batch1.jpg  0.1 MB
  train_batch2.jpg  0.1 MB
  weights\best.pt  409.7 MB
  weights\last.pt  409.7 MB

── D: drive ──
  args.yaml  0.0 MB
  labels.jpg  0.1 MB
  labels_correlogram.jpg  0.2 MB
  results.csv  0.0 MB
  train_batch0.jpg  0.1 MB
  train_batch1.jpg  0.1 MB
  train_batch2.jpg  0.1 MB
  weights\best.pt  409.7 MB
  weights\last.pt  409.7 MB


In [10]:
import shutil
from pathlib import Path

C_WEIGHTS = Path(r"c:\Users\SHANTANU\flir_adas\runs\yolov8x_flir\weights")
D_WEIGHTS = Path(r"D:\flir_adas_runs\runs\yolov8x_flir\weights")

# Copy both weights files from C: to D:
for f in ["last.pt", "best.pt"]:
    src = C_WEIGHTS / f
    dst = D_WEIGHTS / f
    shutil.copy2(src, dst)
    print(f"Copied {f}: {src.stat().st_size/1e6:.1f} MB -> {dst}")

# Verify
import torch
ckpt = torch.load(str(D_WEIGHTS / "last.pt"), map_location="cpu")
print(f"\nD: last.pt epoch: {ckpt.get('epoch', 'unknown')}")

Copied last.pt: 409.7 MB -> D:\flir_adas_runs\runs\yolov8x_flir\weights\last.pt
Copied best.pt: 409.7 MB -> D:\flir_adas_runs\runs\yolov8x_flir\weights\best.pt

D: last.pt epoch: 28


In [9]:
import torch
from pathlib import Path

# Check C: last.pt epoch after restoring
C_RUNS = Path(r"c:\Users\SHANTANU\flir_adas\runs")
ckpt   = torch.load(str(C_RUNS / "yolov8x_flir" / "weights" / "last.pt"),
                    map_location="cpu")
print(f"C: last.pt epoch: {ckpt.get('epoch', 'unknown')}")

C: last.pt epoch: 28


In [10]:
# ======================================================================
# CELL 9 - Evaluate
# ======================================================================
from ultralytics import YOLO
from pathlib import Path

ROOT         = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR     = ROOT / "data"
TRAINING_DIR = ROOT / "training"
NEW_RUNS     = Path(r"D:\flir_adas_runs\runs")

best = NEW_RUNS / "yolov8x_flir" / "weights" / "best.pt"
assert best.exists(), f"Weights not found: {best}"
print(f"Using weights: {best}")

model   = YOLO(str(best))
metrics = model.val(
    data   = str(TRAINING_DIR / "data.yaml"),
    imgsz  = 640,
    batch  = 4,
    device = 0,
    split  = "val",
    plots  = True,
)

names = (DATA_DIR / "names.txt").read_text().strip().splitlines()

print(f"\n{'Metric':<15} {'Value':>8}")
print("-" * 25)
print(f"{'mAP50':<15} {metrics.box.map50:>8.4f}")
print(f"{'mAP50-95':<15} {metrics.box.map:>8.4f}")
print(f"{'Precision':<15} {metrics.box.mp:>8.4f}")
print(f"{'Recall':<15} {metrics.box.mr:>8.4f}")

print(f"\n{'Class':<22} {'AP50':>8} {'AP50-95':>10}  {'Samples':>8}")
print("-" * 55)
for i, (ap50, ap) in enumerate(zip(metrics.box.ap50, metrics.box.ap)):
    note = " ⚠ rare" if names[i] in {"dog", "train"} else ""
    print(f"{names[i]:<22} {ap50:>8.4f} {ap:>10.4f}{note}")

Using weights: D:\flir_adas_runs\runs\yolov8x_flir\weights\best.pt
Ultralytics YOLOv8.2.27  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 268 layers, 68138013 parameters, 0 gradients, 257.5 GFLOPs


val: Scanning C:\Users\SHANTANU\flir_adas\data\labels\val.cache... 3749 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3749/3749 [00:00<?, ?it/s]

val: WARNING  C:\Users\SHANTANU\flir_adas\data\images\val\video-RMxN6a4CcCeLGu4tA-frame-000207-y5eHLa6XNJkpcXEAS.jpg: 1 duplicate labels removed



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 938/938 [18:44<00:00,  1.20s/it]


                   all       3749      84785      0.516       0.35      0.368      0.206
                person       2233      11278       0.65      0.596      0.634      0.365
                  bike        263        446      0.228      0.433      0.234      0.113
                   car       3614      30892      0.649      0.667      0.701      0.487
                 motor        989       3573      0.597      0.576      0.583      0.289
                 truck        791       1499      0.366      0.227      0.158      0.046
                 light       1441      17818       0.64      0.245      0.294      0.127
               hydrant        309        402       0.83      0.231      0.425       0.24
                  sign       2140      17213      0.558      0.419      0.433      0.263
                   dog         17         17          0          0          0          0
         other vehicle        505       1647      0.647      0.103      0.221      0.135
Speed: 0.8ms preproce

In [11]:
# ======================================================================
# CELL 10 - Parse Labelbox Frame Metadata
# ======================================================================
import json
from pathlib import Path

ROOT       = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR   = ROOT / "data"
ANNOT_DIR  = DATA_DIR / "annotations"
RAW_TRAIN  = DATA_DIR / "training_data"

ANNOT_VIDEO = ANNOT_DIR / "training_index.json"
assert ANNOT_VIDEO.exists(), f"Not found: {ANNOT_VIDEO}"

with open(ANNOT_VIDEO, "r", encoding="utf-8") as f:
    frames_json = json.load(f)

disk_images   = list(RAW_TRAIN.glob("*.jpg")) + list(RAW_TRAIN.glob("*.png"))
disk_name_map = {p.stem: p.name for p in disk_images}

frame_meta = {}
for fr in frames_json.get("frames", []):
    video_meta       = fr.get("videoMetadata", {})
    video_id         = video_meta.get("videoId")
    frame_idx        = video_meta.get("frameIndex")
    dataset_frame_id = fr.get("datasetFrameId")
    if not dataset_frame_id:
        continue
    for stem, fname in disk_name_map.items():
        if dataset_frame_id in stem:
            frame_meta[fname] = {"videoId": video_id, "frameIndex": frame_idx}
            break

print(f"Mapped {len(frame_meta)} frames")
for k, v in list(frame_meta.items())[:5]:
    print(f"  {k} -> {v}")

Mapped 10318 frames
  video-23bsd9bsr962GdFBZ-frame-000221-arDqqfMYcrbEEqsex.jpg -> {'videoId': '23bsd9bsr962GdFBZ', 'frameIndex': 221}
  video-23bsd9bsr962GdFBZ-frame-000236-CKrQ59pg7BzJDALSd.jpg -> {'videoId': '23bsd9bsr962GdFBZ', 'frameIndex': 236}
  video-23bsd9bsr962GdFBZ-frame-000251-k8G4q7PbdPFf7Ymr5.jpg -> {'videoId': '23bsd9bsr962GdFBZ', 'frameIndex': 251}
  video-23bsd9bsr962GdFBZ-frame-000266-gKf6k9v5tz6FM4ssf.jpg -> {'videoId': '23bsd9bsr962GdFBZ', 'frameIndex': 266}
  video-23bsd9bsr962GdFBZ-frame-000281-WbvC2cEAjDaym8CGn.jpg -> {'videoId': '23bsd9bsr962GdFBZ', 'frameIndex': 281}


In [13]:
# ======================================================================
# CELL 11 - Tracking + Speed Estimation
# ======================================================================
from ultralytics import YOLO
from collections import defaultdict
from pathlib import Path
import math

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"
RAW_TRAIN = DATA_DIR / "training_data"
NEW_RUNS  = Path(r"D:\flir_adas_runs\runs")

best = NEW_RUNS / "yolov8x_flir" / "weights" / "best.pt"
assert best.exists(), f"Weights not found: {best}"
assert len(frame_meta) > 0, "Run Cell 10 first"

model       = YOLO(str(best))
DEFAULT_FPS = 30.0

def center_xyxy(xyxy):
    x1, y1, x2, y2 = xyxy
    return (x1 + x2) / 2, (y1 + y2) / 2

results = model.track(
    source  = str(RAW_TRAIN),
    tracker = "bytetrack.yaml",
    persist = True,
    stream  = True,
)

tracks = defaultdict(list)
for res in results:
    frame_path = Path(res.path).name
    meta       = frame_meta.get(frame_path)
    if not meta or res.boxes is None:
        continue
    t = meta["frameIndex"] / DEFAULT_FPS
    for box in res.boxes:
        if box.id is None:
            continue
        tid    = int(box.id.item())
        cx, cy = center_xyxy(box.xyxy[0].tolist())
        tracks[tid].append((t, cx, cy, frame_path))

track_speeds = {}
for tid, pts in tracks.items():
    pts    = sorted(pts, key=lambda x: x[0])
    speeds = []
    for (t1, x1, y1, _), (t2, x2, y2, _) in zip(pts, pts[1:]):
        dt = t2 - t1
        if dt <= 0:
            continue
        speed_px_s = math.hypot(x2 - x1, y2 - y1) / dt
        speeds.append((round(t2, 3), round(speed_px_s, 2)))
    track_speeds[tid] = speeds

print(f"Tracked {len(track_speeds)} unique objects\n")
print(f"{'Track ID':<10} {'Samples':>8}  First 3 readings (time_s, px/s)")
print("-" * 55)
for tid, sp in list(track_speeds.items())[:10]:
    print(f"{tid:<10} {len(sp):>8}  {sp[:3]}")

requirements: Ultralytics requirement ['lapx>=0.5.2'] not found, attempting AutoUpdate...

requirements: AutoUpdate success  30.4s, installed 1 package: ['lapx>=0.5.2']
requirements:  Restart runtime or rerun command for updates to take effect


image 1/10319 c:\Users\SHANTANU\flir_adas\data\training_data\video-23bsd9bsr962GdFBZ-frame-000221-arDqqfMYcrbEEqsex.jpg: 576x640 1 sign, 504.8ms
image 2/10319 c:\Users\SHANTANU\flir_adas\data\training_data\video-23bsd9bsr962GdFBZ-frame-000236-CKrQ59pg7BzJDALSd.jpg: 576x640 1 sign, 373.0ms
image 3/10319 c:\Users\SHANTANU\flir_adas\data\training_data\video-23bsd9bsr962GdFBZ-frame-000251-k8G4q7PbdPFf7Ymr5.jpg: 576x640 1 sign, 313.9ms
image 4/10319 c:\Users\SHANTANU\flir_adas\data\training_data\video-23bsd9bsr962GdFBZ-frame-000266-gKf6k9v5tz6FM4ssf.jpg: 576x640 1 sign, 302.4ms
image 5/10319 c:\Users\SHANTANU\flir_adas\data\training_data\video-23bsd9bsr962GdFBZ-frame-000281-WbvC2cEAjDaym8CGn.jpg: 576x640 (no detections), 308.6ms
image 6/10319 c:\Use

In [14]:
# CELL 11b - Save tracking results
import json
from pathlib import Path

OUT = Path(r"D:\flir_adas_runs\tracking_results.json")

# Convert to serializable format
save_data = {
    str(tid): speeds 
    for tid, speeds in track_speeds.items()
}

with open(OUT, "w") as f:
    json.dump(save_data, f, indent=2)

print(f"Saved {len(save_data)} tracks to {OUT}")
print(f"File size: {OUT.stat().st_size/1e6:.1f} MB")

Saved 6747 tracks to D:\flir_adas_runs\tracking_results.json
File size: 2.1 MB


In [ ]:
import json
from pathlib import Path

with open(Path(r"c:\Users\SHANTANU\flir_adas\data\annotations\training_index.json")) as f:
    data = json.load(f)

# Check frame dimensions
frames = data.get("frames", [])
widths  = set()
heights = set()
for fr in frames[:100]:
    widths.add(fr.get("width"))
    heights.add(fr.get("height"))

print(f"Frame widths  : {widths}")
print(f"Frame heights : {heights}")

# Check video descriptions for camera info
print("\nVideo descriptions:")
videos = data.get("videos", [])
for v in videos[:5]:
    print(f"  {v['name']:<30} {v.get('description','')[:80]}")

In [ ]:
# # === Cell 4: Sanity check — every image must have a paired label ===
# from pathlib import Path

# def check_pairing(images_dir: Path, labels_dir: Path, split: str):
#     imgs = sorted(p for p in images_dir.iterdir()
#                   if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
#     paired   = [p for p in imgs if (labels_dir / p.with_suffix(".txt").name).exists()]
#     unpaired = [p for p in imgs if p not in paired]

#     print(f"\n── {split} ──────────────────────────────")
#     print(f"  Images total   : {len(imgs)}")
#     print(f"  Images w/ label: {len(paired)}")
#     print(f"  Images w/o label (backgrounds): {len(unpaired)}")
#     if unpaired:
#         print(f"  First 5 unpaired: {[p.name for p in unpaired[:5]]}")

#     # Quick peek at a label file
#     sample_labels = list(labels_dir.glob("*.txt"))[:3]
#     for lbl in sample_labels:
#         lines = lbl.read_text().strip().splitlines()
#         print(f"  Sample {lbl.name}: {lines[:2]}")

#     assert len(paired) > 0, (
#         f"FATAL: 0 paired images in {split}! "
#         "Check that label stems match image stems exactly."
#     )

# check_pairing(TRAIN_IMAGES_DIR, LABELS_TRAIN, "TRAIN")
# check_pairing(TEST_IMAGES_DIR,  LABELS_TEST,  "VAL")
# print("\n✓ Pairing check passed — safe to train.")


# ======================================================================
# CELL 4 - Remap Class Indices (remove unused classes)
# ======================================================================
from pathlib import Path
from collections import Counter
from tqdm import tqdm

names_all = (DATA_DIR / "names.txt").read_text().strip().splitlines()

# Scan all label files to find used class indices
counter = Counter()
for ldir in [LBL_TRAIN, LBL_VAL]:
    for lf in ldir.glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1

used_indices = sorted(counter.keys())
new_names    = [names_all[i] for i in used_indices]
old_to_new   = {old: new for new, old in enumerate(used_indices)}

print(f"Classes used: {len(used_indices)} of {len(names_all)}")
print(f"\n{'old':>5}  {'new':>5}  {'name':<22}  {'count':>8}")
print("-" * 48)
for old, new in old_to_new.items():
    print(f"{old:>5}  {new:>5}  {names_all[old]:<22}  {counter[old]:>8}")

# Rewrite label files with remapped indices
def remap_labels(labels_dir, old_to_new, split):
    files = list(labels_dir.glob("*.txt"))
    for lf in tqdm(files, desc=f"Remapping {split}"):
        lines     = lf.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.split()
            if not parts:
                continue
            old_cls = int(parts[0])
            if old_cls not in old_to_new:
                continue
            parts[0] = str(old_to_new[old_cls])
            new_lines.append(" ".join(parts))
        lf.write_text("\n".join(new_lines) + "\n", encoding="utf-8")

remap_labels(LBL_TRAIN, old_to_new, "train")
remap_labels(LBL_VAL,   old_to_new, "val")

# Update names.txt
names = new_names
(DATA_DIR / "names.txt").write_text("\n".join(names), encoding="utf-8")
print(f"\nFinal classes ({len(names)}): {names}")


In [ ]:
# # === Cell 5: Write data.yaml ===
# # YOLO auto-discovers labels by replacing 'images' with 'labels' in the path.
# # That's why the folder structure from Cell 2 matters.
# import yaml, json

# # Reload names in case Cell 3 was re-run independently
# names_file = DATA_DIR / "names.txt"
# if names_file.exists():
#     names = names_file.read_text().strip().splitlines()
# else:
#     raise FileNotFoundError("names.txt missing — re-run Cell 3 first.")

# data_yaml_content = {
#     "path" : str(DATA_DIR.resolve()),   # dataset root
#     "train": "images/train",            # relative to 'path'
#     "val"  : "images/val",              # relative to 'path'
#     "nc"   : len(names),
#     "names": names,
# }

# yaml_path = ROOT / "data.yaml"
# with open(yaml_path, "w") as f:
#     yaml.dump(data_yaml_content, f, sort_keys=False)

# print("Wrote data.yaml:")
# print(yaml_path.read_text())


# ======================================================================
# CELL 5 - Fix Image Folders (remove junctions, create real dirs)
# ======================================================================
import os, ctypes, shutil
from pathlib import Path
from tqdm import tqdm

def is_junction(p: Path) -> bool:
    try:
        attrs = ctypes.windll.kernel32.GetFileAttributesW(str(p))
        return bool(attrs & 0x400)
    except Exception:
        return False

def remove_junction_or_dir(p: Path):
    if not p.exists() and not p.is_symlink():
        print(f"  Does not exist (ok): {p.name}")
        return
    if is_junction(p):
        os.rmdir(p)
        print(f"  Removed junction   : {p.name}")
    else:
        shutil.rmtree(p)
        print(f"  Removed real dir   : {p.name}")

def hardlink_images(src: Path, dst: Path, label: str):
    exts  = {".jpg", ".jpeg", ".png"}
    imgs  = [p for p in src.iterdir() if p.suffix.lower() in exts]
    count = 0
    for img in tqdm(imgs, desc=f"  Linking {label}"):
        dst_file = dst / img.name
        if not dst_file.exists():
            try:
                os.link(img, dst_file)
            except OSError:
                shutil.copy2(img, dst_file)
            count += 1
    print(f"  {label}: {count} images linked -> {dst}")

# Remove old junctions
print("── Step 1: Remove junctions ──")
remove_junction_or_dir(IMG_TRAIN)
remove_junction_or_dir(IMG_VAL)

# Create real directories
print("\n── Step 2: Create real dirs ──")
IMG_TRAIN.mkdir(parents=True, exist_ok=True)
IMG_VAL.mkdir(parents=True, exist_ok=True)
print(f"  Created: {IMG_TRAIN}")
print(f"  Created: {IMG_VAL}")

# Hardlink images
print("\n── Step 3: Hardlink images ──")
hardlink_images(RAW_TRAIN, IMG_TRAIN, "TRAIN")
hardlink_images(RAW_TEST,  IMG_VAL,   "VAL")

# Verify
print("\n── Step 4: Verify ──")
for p, label in [(IMG_TRAIN, "TRAIN"), (IMG_VAL, "VAL")]:
    junc     = is_junction(p)
    resolved = str(p.resolve()).replace("\\", "/")
    count    = len(list(p.iterdir()))
    print(f"  {label}")
    print(f"    is_junction : {junc}      <- must be False")
    print(f"    resolves to : {resolved}  <- must contain 'images'")
    print(f"    file count  : {count}")
    assert not junc,                  f"Junction still active: {p}"
    assert "images" in resolved,      f"Wrong resolved path: {resolved}"
    assert count > 0,                 f"No images in: {p}"

print("\n✓ Image folders are real directories")


In [ ]:
# ======================================================================
# CELL 6 - Verify Everything Before Training
# ======================================================================
from pathlib import Path
from collections import Counter

# 1. Stem matching
def check_stems(imgs_dir, lbls_dir, split):
    imgs   = {p.stem for p in imgs_dir.glob("*")
              if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
    labels = {p.stem for p in lbls_dir.glob("*.txt")}
    matched = imgs & labels
    print(f"\n── {split} ──────────────────────────────")
    print(f"  images  : {len(imgs)}")
    print(f"  labels  : {len(labels)}")
    print(f"  matched : {len(matched)}  <- must be > 0")
    print(f"  backgrounds (no label): {len(imgs - labels)}")
    assert len(matched) > 0, f"0 matched pairs in {split}!"
    return len(matched)

check_stems(IMG_TRAIN, LBL_TRAIN, "TRAIN")
check_stems(IMG_VAL,   LBL_VAL,   "VAL")

# 2. YOLO path swap check
names  = (DATA_DIR / "names.txt").read_text().strip().splitlines()
train_resolved = str(IMG_TRAIN.resolve()).replace("\\", "/")
lbl_via_swap   = train_resolved.replace("images", "labels")
lbl_actual     = str(LBL_TRAIN.resolve()).replace("\\", "/")

print(f"\n── YOLO label discovery check ──────────")
print(f"  YOLO swaps path to : {lbl_via_swap}")
print(f"  Labels actually at : {lbl_actual}")
print(f"  Match              : {lbl_via_swap == lbl_actual}  <- must be True")
assert lbl_via_swap == lbl_actual, (
    f"YOLO will not find labels!\n"
    f"  Expected: {lbl_via_swap}\n"
    f"  Actual  : {lbl_actual}"
)

# 3. Class index check
ctr = Counter()
for ldir in [LBL_TRAIN, LBL_VAL]:
    for lf in ldir.glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                ctr[int(line.split()[0])] += 1
max_idx = max(ctr.keys())
print(f"\n── Class check ─────────────────────────")
print(f"  nc        : {len(names)}")
print(f"  max index : {max_idx}  <- must be < nc")
assert max_idx < len(names), f"Class index {max_idx} out of range for nc={len(names)}"

print("\n✓ ALL CHECKS PASSED — safe to write data.yaml and train")


In [ ]:
# # === STEP 1: Audit what actually exists on disk ===
# from pathlib import Path

# ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
# DATA_DIR = ROOT / "data"

# print("=== Folder tree under data/ ===")
# for p in sorted(DATA_DIR.rglob("*")):
#     if p.is_dir():
#         n_files = len(list(p.iterdir()))
#         print(f"  DIR  {p.relative_to(ROOT)}  ({n_files} items)")

# print("\n=== Checking critical paths ===")
# for path in [
#     DATA_DIR / "training_data",
#     DATA_DIR / "test_data",
#     DATA_DIR / "images" / "train",
#     DATA_DIR / "images" / "val",
#     DATA_DIR / "labels" / "train",
#     DATA_DIR / "labels" / "val",
#     ROOT / "data.yaml",
# ]:
#     exists = path.exists()
#     count  = ""
#     if exists and path.is_dir():
#         items = list(path.iterdir())
#         count = f"  → {len(items)} items"
#     print(f"  {'✓' if exists else '✗'}  {path.relative_to(ROOT)}{count}")


# ======================================================================
# CELL 7 - Write data.yaml
# ======================================================================
from pathlib import Path

names     = (DATA_DIR / "names.txt").read_text().strip().splitlines()
TRAIN_ABS = str(IMG_TRAIN.resolve()).replace("\\", "/")
VAL_ABS   = str(IMG_VAL.resolve()).replace("\\", "/")

# Delete all stale caches
for c in ROOT.rglob("*.cache"):
    c.unlink()
    print(f"Deleted cache: {c.name}")

# Write yaml as plain text — no library, no formatting surprises
name_lines = "\n".join(f"  - {n}" for n in names)
yaml_text  = (
    f"train: {TRAIN_ABS}\n"
    f"val: {VAL_ABS}\n"
    f"nc: {len(names)}\n"
    f"names:\n{name_lines}\n"
)

yaml_path = TRAINING_DIR / "data.yaml"
yaml_path.write_text(yaml_text, encoding="utf-8")

print("\n✓ data.yaml written:")
print(yaml_path.read_text())

# Confirm
assert "images/train" in yaml_path.read_text(), "images/train not in yaml!"
assert "images/val"   in yaml_path.read_text(), "images/val not in yaml!"
print("✓ data.yaml verified")


In [ ]:
from pathlib import Path
ROOT = Path(r"c:\Users\SHANTANU\flir_adas")

print(open(ROOT / "data.yaml").read())

In [ ]:
from pathlib import Path

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"

TRAIN_IMGS = DATA_DIR / "images" / "train"
VAL_IMGS   = DATA_DIR / "images" / "val"
LBL_TRAIN  = DATA_DIR / "labels" / "train"
LBL_VAL    = DATA_DIR / "labels" / "val"

def stem_report(imgs_dir, lbls_dir, split):
    imgs   = {p.stem: p.name for p in imgs_dir.glob("*")
              if p.suffix.lower() in {".jpg",".jpeg",".png"}}
    labels = {p.stem: p.name for p in lbls_dir.glob("*.txt")}
    matched   = set(imgs) & set(labels)
    unmatched = set(imgs) - set(labels)

    print(f"\n── {split} ───────────────────────────────────")
    print(f"  Images : {len(imgs)}")
    print(f"  Labels : {len(labels)}")
    print(f"  Matched: {len(matched)}  ← this must be > 0")
    print(f"  Images without labels: {len(unmatched)}")

    # Show samples so we can spot naming differences
    sample_imgs   = sorted(imgs.values())[:5]
    sample_labels = sorted(labels.values())[:5]
    print(f"  Sample image  names : {sample_imgs}")
    print(f"  Sample label  names : {sample_labels}")

stem_report(TRAIN_IMGS, LBL_TRAIN, "TRAIN")
stem_report(VAL_IMGS,   LBL_VAL,   "VAL")

In [ ]:
from pathlib import Path

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"

# Check actual class count in names.txt
names = (DATA_DIR / "names.txt").read_text().strip().splitlines()
print(f"nc = {len(names)}")
print("Classes:", names)

# Spot-check a few label files to see what class indices appear
import random
label_files = list((DATA_DIR / "labels" / "train").glob("*.txt"))
sample = random.sample(label_files, min(10, len(label_files)))

all_class_ids = set()
for lf in sample:
    for line in lf.read_text().strip().splitlines():
        if line:
            all_class_ids.add(int(line.split()[0]))

print(f"\nClass indices seen in sample labels: {sorted(all_class_ids)}")
print(f"Max class index: {max(all_class_ids)}")
print(f"nc in yaml     : {len(names)}")
print(f"Match OK       : {max(all_class_ids) < len(names)}")


In [ ]:
from pathlib import Path
from collections import Counter

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"

names_all = (DATA_DIR / "names.txt").read_text().strip().splitlines()

label_dirs = [
    DATA_DIR / "labels" / "train",
    DATA_DIR / "labels" / "val",
]

class_counter = Counter()

for ldir in label_dirs:
    for lf in ldir.glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                class_counter[int(line.split()[0])] += 1

print("Class index → name → annotation count:")
print(f"{'idx':>5}  {'name':<20}  {'count':>8}")
print("-" * 40)
for idx in sorted(class_counter):
    print(f"{idx:>5}  {names_all[idx]:<20}  {class_counter[idx]:>8}")

used_indices = sorted(class_counter.keys())
print(f"\nTotal classes actually used: {len(used_indices)}")
print(f"Used indices: {used_indices}")

In [ ]:
import yaml, shutil
from pathlib import Path
from collections import Counter
from tqdm import tqdm

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"

names_all = (DATA_DIR / "names.txt").read_text().strip().splitlines()

# ── Collect used indices from full scan ──────────────────────────────────────
class_counter = Counter()
for ldir in [DATA_DIR/"labels"/"train", DATA_DIR/"labels"/"val"]:
    for lf in ldir.glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                class_counter[int(line.split()[0])] += 1

used_indices  = sorted(class_counter.keys())
new_names     = [names_all[i] for i in used_indices]
old_to_new    = {old: new for new, old in enumerate(used_indices)}

print(f"Remapping {len(used_indices)} classes:")
for old, new in old_to_new.items():
    print(f"  {old:>3} {names_all[old]:<20} → {new}")

# ── Rewrite all label files in-place ─────────────────────────────────────────
def remap_labels(labels_dir: Path, old_to_new: dict, split: str):
    files = list(labels_dir.glob("*.txt"))
    changed = 0
    for lf in tqdm(files, desc=f"Remapping {split}"):
        lines = lf.read_text().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.split()
            if not parts: continue
            old_cls = int(parts[0])
            if old_cls not in old_to_new:
                continue  # drop annotations for unused classes (shouldn't happen)
            parts[0] = str(old_to_new[old_cls])
            new_lines.append(" ".join(parts))
        lf.write_text("\n".join(new_lines) + "\n" if new_lines else "")
        changed += 1
    print(f"  {split}: rewrote {changed} label files")

remap_labels(DATA_DIR / "labels" / "train", old_to_new, "train")
remap_labels(DATA_DIR / "labels" / "val",   old_to_new, "val")

# ── Save trimmed names.txt ────────────────────────────────────────────────────
(DATA_DIR / "names.txt").write_text("\n".join(new_names))
print(f"\nNew names.txt ({len(new_names)} classes): {new_names}")

# ── Delete stale cache ────────────────────────────────────────────────────────
for c in DATA_DIR.rglob("*.cache"):
    c.unlink()
    print(f"Deleted cache: {c.name}")

# ── Rewrite data.yaml ─────────────────────────────────────────────────────────
yaml_content = {
    "path" : str(DATA_DIR.resolve()),
    "train": "images/train",
    "val"  : "images/val",
    "nc"   : len(new_names),
    "names": new_names,
}
yaml_path = ROOT / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"\n✓ data.yaml updated:")
print(yaml_path.read_text())

In [ ]:
from pathlib import Path
from collections import Counter

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"

names = (DATA_DIR / "names.txt").read_text().strip().splitlines()
print(f"nc = {len(names)}  →  {names}\n")

# Re-scan to confirm indices are now 0..N-1
counter = Counter()
for ldir in [DATA_DIR/"labels"/"train", DATA_DIR/"labels"/"val"]:
    for lf in ldir.glob("*.txt"):
        for line in lf.read_text().strip().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1

print(f"{'idx':>4}  {'name':<20}  {'count':>8}")
print("-"*38)
for idx in sorted(counter):
    name = names[idx] if idx < len(names) else "OUT-OF-RANGE ⚠"
    print(f"{idx:>4}  {name:<20}  {counter[idx]:>8}")

max_idx = max(counter)
print(f"\nMax class index : {max_idx}")
print(f"nc              : {len(names)}")
print(f"Valid           : {'✓ YES' if max_idx < len(names) else '✗ NO — fix before training!'}")

In [ ]:
from ultralytics.data.utils import check_det_dataset
from pathlib import Path

ROOT = Path(r"c:\Users\SHANTANU\flir_adas")

# This is what YOLO internally does before training — if this passes, training will work
result = check_det_dataset(str(ROOT / "data.yaml"))
print("Dataset check result:")
print(f"  train path : {result['train']}")
print(f"  val path   : {result['val']}")
print(f"  nc         : {result['nc']}")
print(f"  names      : {result['names']}")

In [ ]:
from pathlib import Path
ROOT = Path(r"c:\Users\SHANTANU\flir_adas")
print(repr(open(ROOT / "data.yaml").read()))

In [ ]:
from pathlib import Path

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"

names = (DATA_DIR / "names.txt").read_text().strip().splitlines()

# Build YAML manually — exact format YOLO expects
name_lines = "\n".join(f"  - {n}" for n in names)

yaml_text = f"""path: {DATA_DIR.resolve()}
train: images/train
val: images/val
nc: {len(names)}
names:
{name_lines}
"""

yaml_path = ROOT / "data.yaml"
yaml_path.write_text(yaml_text, encoding="utf-8")

print("=== data.yaml written ===")
print(yaml_path.read_text())

In [ ]:
from pathlib import Path

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"

IMG_TRAIN = DATA_DIR / "images" / "train"
LBL_TRAIN = DATA_DIR / "labels" / "train"

# Check 1: is images/train a real dir or junction?
import ctypes
attrs     = ctypes.windll.kernel32.GetFileAttributesW(str(IMG_TRAIN))
is_reparse = bool(attrs & 0x400)
print(f"images/train is junction : {is_reparse}")  # must be False after Cell 5
print(f"images/train resolves to : {IMG_TRAIN.resolve()}")  # must contain 'images'

# Check 2: do stems match?
imgs   = {p.stem for p in IMG_TRAIN.glob("*") if p.suffix.lower() in {".jpg",".png"}}
labels = {p.stem for p in LBL_TRAIN.glob("*.txt")}
matched = imgs & labels
print(f"\nImages  : {len(imgs)}")
print(f"Labels  : {len(labels)}")
print(f"Matched : {len(matched)}")  # must be > 0

# Check 3: simulate YOLO path swap
yolo_lbl_path = str(IMG_TRAIN.resolve()).replace("images", "labels")
print(f"\nYOLO expects labels at : {yolo_lbl_path}")
print(f"Labels actually at     : {LBL_TRAIN.resolve()}")
print(f"Paths match            : {yolo_lbl_path == str(LBL_TRAIN.resolve())}")  # must be True

In [ ]:
from pathlib import Path
from ultralytics.data.utils import check_det_dataset

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"

# Nuke every cache
for c in ROOT.rglob("*.cache"):
    print(f"Deleted: {c}")
    c.unlink()

# Verify YOLO reads the right paths now
result = check_det_dataset(str(ROOT / "data.yaml"))
print(f"\ntrain path : {result['train']}")
print(f"val path   : {result['val']}")
print(f"nc         : {result['nc']}")

assert "images" in str(result['train']), \
    f"STILL WRONG: {result['train']}"
assert "images" in str(result['val']), \
    f"STILL WRONG: {result['val']}"

print("\n✓ data.yaml is correct — safe to train!")

In [ ]:
# === COMPLETE RESET CELL — run this once, then train immediately after ===
from pathlib import Path
import yaml, os

ROOT     = Path(r"c:\Users\SHANTANU\flir_adas")
DATA_DIR = ROOT / "data"

# ── 1. Delete every cache and every yaml except what we create ────────────────
print("── Cleaning stale files ──")
for f in ROOT.rglob("*.cache"):
    f.unlink(); print(f"  deleted cache : {f.relative_to(ROOT)}")

# Delete any yaml files inside runs/ that might confuse YOLO
for f in (ROOT / "runs").rglob("*.yaml") if (ROOT / "runs").exists() else []:
    print(f"  found runs yaml: {f}")  # just report, don't delete yet

# ── 2. Verify images and labels exist ─────────────────────────────────────────
print("\n── Path verification ──")
paths_to_check = {
    "images/train" : DATA_DIR / "images" / "train",
    "images/val"   : DATA_DIR / "images" / "val",
    "labels/train" : DATA_DIR / "labels" / "train",
    "labels/val"   : DATA_DIR / "labels" / "val",
}
for label, p in paths_to_check.items():
    items = len(list(p.iterdir())) if p.exists() else 0
    status = "✓" if p.exists() and items > 0 else "✗ MISSING"
    print(f"  {status}  {label:<20} → {items} items")

# ── 3. Load and show class names ──────────────────────────────────────────────
names = (DATA_DIR / "names.txt").read_text().strip().splitlines()
print(f"\n── Classes: {len(names)} ──")
print(names)

# ── 4. Write data.yaml with FULLY ABSOLUTE paths — no relative paths at all ──
print("\n── Writing data.yaml ──")

# Absolute paths, forward slashes (YOLO handles both on Windows)
TRAIN_ABS = str((DATA_DIR / "images" / "train").resolve()).replace("\\", "/")
VAL_ABS   = str((DATA_DIR / "images" / "val").resolve()).replace("\\", "/")

# Build name list lines
name_lines = "\n".join(f"  - {n}" for n in names)

# Write as plain text — no yaml library involved (eliminates any serialization bugs)
yaml_content = f"train: {TRAIN_ABS}\nval: {VAL_ABS}\nnc: {len(names)}\nnames:\n{name_lines}\n"

yaml_path = ROOT / "data.yaml"
yaml_path.write_text(yaml_content, encoding="utf-8")

print(yaml_path.read_text())

# ── 5. Sanity check: read it back raw ─────────────────────────────────────────
raw = yaml_path.read_text()
assert "images/train" in raw, "images/train not found in yaml!"
assert "images/val"   in raw, "images/val not found in yaml!"
assert "training_data" not in raw, "OLD PATH still in yaml!"
assert "test_data"     not in raw, "OLD PATH still in yaml!"
print("✓ data.yaml content verified — no old paths present")

# ── 6. Parse it back with yaml to confirm structure ───────────────────────────
parsed = yaml.safe_load(raw)
print(f"\nParsed train : {parsed['train']}")
print(f"Parsed val   : {parsed['val']}")
print(f"Parsed nc    : {parsed['nc']}")
assert parsed["train"] == TRAIN_ABS, f"Parsed train mismatch: {parsed['train']}"
assert parsed["val"]   == VAL_ABS,   f"Parsed val mismatch: {parsed['val']}"
print("\n✓ All checks passed — ready to train!")

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

ROOT = Path(r"c:\Users\SHANTANU\flir_adas")

for c in (ROOT / "data").rglob("*.cache"):
    c.unlink()

print("GPU:", torch.cuda.get_device_name(0))

model = YOLO(str(ROOT / "yolov8x.pt"))
model.train(
    data     = str(ROOT / "data.yaml"),
    epochs   = 50,
    imgsz    = 640,
    batch    = 2,
    device   = 0,
    project  = str(ROOT / "runs"),
    name     = "yolov8x_flir",
    exist_ok = True,
    workers  = 2,
    amp      = True,
    cache    = False,
)

best = ROOT / "runs" / "yolov8x_flir" / "weights" / "best.pt"
print("✓ Done. Best weights:", best)

In [ ]:
# === Cell 7: Parse Labelbox index.json -> frame metadata ===
import json
from pathlib import Path

ANNOT_VIDEO = ANNOT_DIR / "training_index.json"
IMAGES_DIR  = TRAIN_IMAGES_DIR   # already points to data/images/train

with open(ANNOT_VIDEO, "r", encoding="utf-8") as f:
    frames_json = json.load(f)

disk_images  = list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.png"))
disk_name_map = {p.stem: p.name for p in disk_images}

frame_meta = {}
for fr in frames_json.get("frames", []):
    video_meta      = fr.get("videoMetadata", {})
    video_id        = video_meta.get("videoId")
    frame_idx       = video_meta.get("frameIndex")
    dataset_frame_id = fr.get("datasetFrameId")
    if not dataset_frame_id:
        continue
    for stem, fname in disk_name_map.items():
        if dataset_frame_id in stem:
            frame_meta[fname] = {"videoId": video_id, "frameIndex": frame_idx}
            break

print(f"Mapped {len(frame_meta)} frames")
for k, v in list(frame_meta.items())[:5]:
    print(f"  {k} -> {v}")

In [ ]:
# === Cell 8: Tracking + speed estimation ===
from ultralytics import YOLO
from collections import defaultdict
from pathlib import Path
import math

MODEL_PATH = OUTPUT_RUNS / "yolov8x_flir" / "weights" / "best.pt"
model      = YOLO(str(MODEL_PATH))
SOURCE_DIR = TRAIN_IMAGES_DIR
DEFAULT_FPS = 30.0

def center_xyxy(xyxy):
    x1, y1, x2, y2 = xyxy
    return (x1 + x2) / 2, (y1 + y2) / 2

results = model.track(
    source  = str(SOURCE_DIR),
    tracker = "bytetrack.yaml",
    persist = True,
    stream  = True,
)

tracks = defaultdict(list)
for res in results:
    frame_path = Path(res.path).name
    meta = frame_meta.get(frame_path)
    if not meta or res.boxes is None:
        continue
    t = meta["frameIndex"] / DEFAULT_FPS
    for box in res.boxes:
        if box.id is None: continue
        tid = int(box.id.item())
        cx, cy = center_xyxy(box.xyxy[0].tolist())
        tracks[tid].append((t, cx, cy, frame_path))

track_speeds = {}
for tid, pts in tracks.items():
    pts = sorted(pts, key=lambda x: x[0])
    speeds = []
    for (t1, x1, y1, _), (t2, x2, y2, _) in zip(pts, pts[1:]):
        dt = t2 - t1
        if dt <= 0: continue
        speed_px_s = math.hypot(x2 - x1, y2 - y1) / dt
        speeds.append((t2, speed_px_s))
    track_speeds[tid] = speeds

for tid, sp in list(track_speeds.items())[:5]:
    print(f"Track {tid}: {sp[:5]}")

# train/val split

In [ ]:
from pathlib import Path

def summarize_labels(images_dir: Path, labels_dir: Path):
    images = sorted([p for p in images_dir.glob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]])
    labels = sorted([p for p in labels_dir.glob("*.txt")])
    imgs_with_labels = [p for p in images if (labels_dir / p.with_suffix('.txt').name).exists()]
    imgs_without_labels = [p for p in images if not (labels_dir / p.with_suffix('.txt').name).exists()]
    extraneous_labels = [p for p in labels if not (images_dir / p.with_suffix('.jpg').name).exists() and not (images_dir / p.with_suffix('.png').name).exists()]

    return {
        "images_total": len(images),
        "labels_total": len(labels),
        "images_with_labels": len(imgs_with_labels),
        "images_without_labels": len(imgs_without_labels),
        "extraneous_label_files": len(extraneous_labels),
        "examples_missing": imgs_without_labels[:5],
        "examples_extraneous": extraneous_labels[:5]
    }

print("TRAIN summary:")
train_summary = summarize_labels(TRAIN_IMAGES_DIR, LABELS_TRAIN)
print(train_summary)

print("\nTEST summary:")
test_summary = summarize_labels(TEST_IMAGES_DIR, LABELS_TEST)
print(test_summary)

# Quick sanity: sample a few label files
print("\nSample label files (train):", list(LABELS_TRAIN.glob("*.txt"))[:5])
print("Sample label files (test):", list(LABELS_TEST.glob("*.txt"))[:5])


# data.yaml for yolo

In [ ]:
# Cell 5: write YOLO-style data YAML
data_yaml = {
    'train': str(TRAIN_IMAGES_DIR.resolve()),
    'val': str(TEST_IMAGES_DIR.resolve()),
    'nc': len(names),
    'names': names
}

import yaml
with open(ROOT / "data.yaml", "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)
print("Wrote:", ROOT / "data.yaml")


# finetuning yolov8x

In [ ]:
# Cell 6: Train YOLOv8x using GPU (CUDA)

from ultralytics import YOLO
import torch

# ---- Sanity check (MUST pass for GPU) ----
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# ---- Load pretrained weights ----
model = YOLO(str(ROOT / "yolov8x.pt"))

# ---- Train (fine-tune) with Safety Net ----
try:
    print("Starting training with AutoBatch (batch=-1) to prevent OOM...")
    model.train(
        data=str(ROOT / "data.yaml"),   # dataset config
        epochs=30,                      # increase if GPU allows
        imgsz=640,
        batch=-1,                       # AutoBatch feature (tries to find safest size)
        device=0,                       # GPU 0 (IMPORTANT)
        project=str(OUTPUT_RUNS),       # runs/
        name="yolov8x_finetune",
        exist_ok=True,
        workers=4,                      # Reduced default workers for safety
        amp=True                        # mixed precision (faster, less VRAM)
    )

    # ---- Output ----
    print("Training complete.")
    print("Best weights saved at:")
    print(OUTPUT_RUNS / "yolov8x_finetune" / "weights" / "best.pt")

except RuntimeError as e:
    # Check if the crash is memory related
    if "out of memory" in str(e).lower() or "oom" in str(e).lower():
        print("====== OOM CRASH CAUGHT ======")
        print("Detailed Report:")
        print("1. CUDA Out Of Memory Error occurred. Your VRAM is too small for this configuration.")
        print(f"2. Exception Details: {e}")
        print("3. AutoBatch (batch=-1) failed to find a safe size, meaning even batch=1 is too large, or system memory choked.")
        print("\nWhat should be changed:")
        print("- Manually set `batch=2` or `batch=4` instead of `-1`.")
        print("- Decrease image size `imgsz` (e.g., to 320 or 480).")
        print("- Use a lighter model like `yolov8m.pt` or `yolov8s.pt` instead of `yolov8x.pt`.")
        print("- Set `workers=2` or `workers=0`.")
        print("================================")
        # Empty cache so the notebook kernel doesn't completely die and you can retry
        torch.cuda.empty_cache()
    else:
        # If it's another kind of error, raise it normally
        raise e


# tracking distance across frames

In [ ]:
# === Cell 7: Parse Labelbox index.json -> filename -> (videoId, frameIndex) ===
import json
from pathlib import Path

# Choose which index to load (training or test)
ANNOT_VIDEO = ANNOT_DIR / "training_index.json"   # change to test_index.json if needed
IMAGES_DIR = DATA_DIR / "training_data"            # or test_data

with open(ANNOT_VIDEO, "r", encoding="utf-8") as f:
    frames_json = json.load(f)

# Build a lookup of actual filenames on disk
disk_images = list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.png"))
disk_name_map = {p.stem: p.name for p in disk_images}

frame_meta = {}

for fr in frames_json.get("frames", []):
    video_meta = fr.get("videoMetadata", {})
    video_id = video_meta.get("videoId")
    frame_idx = video_meta.get("frameIndex")

    # Labelbox usually stores datasetFrameId which is embedded in filename
    dataset_frame_id = fr.get("datasetFrameId")

    if not dataset_frame_id:
        continue

    # Try to match datasetFrameId to actual disk filenames
    for stem, fname in disk_name_map.items():
        if dataset_frame_id in stem:
            frame_meta[fname] = {
                "videoId": video_id,
                "frameIndex": frame_idx
            }
            break

print("Mapped frame metadata entries:", len(frame_meta))

# Sanity check
sample = list(frame_meta.items())[:5]
for k, v in sample:
    print(k, "->", v)


In [ ]:
# === Cell 8: Tracking + speed estimation (pixel/sec) ===
from ultralytics import YOLO
from collections import defaultdict
from pathlib import Path
import math

# Load trained model
MODEL_PATH = OUTPUT_RUNS / "yolov8x_finetune" / "weights" / "best.pt"
model = YOLO(str(MODEL_PATH))

# Image source (training or test)
SOURCE_DIR = DATA_DIR / "training_data"   # or test_data

DEFAULT_FPS = 30.0  # replace if you have per-video FPS

def center_xyxy(xyxy):
    x1, y1, x2, y2 = xyxy
    return ((x1 + x2) / 2, (y1 + y2) / 2)

# Run tracker
results = model.track(
    source=str(SOURCE_DIR),
    tracker="bytetrack.yaml",
    persist=True,
    stream=True
)

# Collect per-track positions
tracks = defaultdict(list)

for res in results:
    frame_path = Path(res.path).name
    meta = frame_meta.get(frame_path)

    if not meta:
        continue

    frame_idx = meta["frameIndex"]
    t = frame_idx / DEFAULT_FPS

    if res.boxes is None:
        continue

    for box in res.boxes:
        if box.id is None:
            continue

        tid = int(box.id.item())
        xyxy = box.xyxy[0].tolist()
        cx, cy = center_xyxy(xyxy)

        tracks[tid].append((t, cx, cy, frame_path))

# Compute speeds
track_speeds = {}

for tid, pts in tracks.items():
    pts = sorted(pts, key=lambda x: x[0])
    speeds = []

    for (t1, x1, y1, _), (t2, x2, y2, _) in zip(pts, pts[1:]):
        dt = t2 - t1
        if dt <= 0:
            continue

        dist_px = math.hypot(x2 - x1, y2 - y1)
        speed_px_s = dist_px / dt
        speeds.append((t2, speed_px_s))

    track_speeds[tid] = speeds

# Print sample output
for tid, sp in list(track_speeds.items())[:5]:
    print(f"Track {tid} speed samples:", sp[:5])
